# TP4 · Analyse de corpus
**Anwar Ezzirani · 5e année · Groupe G4**
**Chapitre 2 · Partie 4 · Explorer un corpus** — Traitement Automatique du Langage Naturel

**Objectif :** décrire le corpus fil rouge (avis de restaurants) avec des chiffres et des graphiques :
fréquences, mots vides de domaine, loi de Zipf, n-grammes, collocations (PMI et LLR), concordances,
nuage de mots, puis rédiger une **courte synthèse**.

**Livrable :** ce notebook complété, avec une courte étude descriptive du corpus (partie 8) :
chiffres clés, courbe de Zipf, 10 collocations commentées.

| Partie | Contenu | Notions du cours |
|---|---|---|
| 0 | Mise en place : bibliothèques, corpus, `tokeniser()` | prétraitement (TP2) |
| 1 | Compter : tokens, types, hapax, TTR | TTR = V / N |
| 2 | Fréquences, mots vides généraux et de domaine | `Counter`, fréquence documentaire (df) |
| 3 | La loi de Zipf et la courbe en log-log | f × r ≈ C |
| 4 | Les n-grammes | N − n + 1 n-grammes |
| 5 | Les collocations : PMI, LLR, NLTK | PMI, G² |
| 6 | Les concordances (KWIC) | `nltk.Text.concordance()` |

### Comment travailler
- Exécutez les cellules **dans l'ordre** (Maj + Entrée).
- Les cellules marquées **`# À COMPLÉTER`** sont à compléter : remplacez les `None` par votre code.
- Après chaque exercice, une cellule **Vérification** vous dit si votre réponse est correcte (✅) ou ce qu'il faut corriger (❌).
- Les questions **✍️** demandent une réponse rédigée en une ou deux phrases.
- Sur Colab, tout est déjà installé. Sur votre machine : `pip install nltk matplotlib wordcloud`.

## Partie 0 · Mise en place

### 0.1 Les bibliothèques

In [ ]:
import re
import math
from collections import Counter

import matplotlib.pyplot as plt
import nltk
from nltk.corpus import stopwords
from nltk.text import Text
from nltk.collocations import BigramCollocationFinder, BigramAssocMeasures

try:
    nltk.data.find("corpora/stopwords")
except LookupError:
    nltk.download("stopwords", quiet=True)

# europarl_raw n'est nécessaire que pour l'extension facultative absente de ce notebook.
print("Bibliothèques chargées.")

### 0.2 L'outil de vérification
Cette fonction vérifie vos réponses. Vous n'avez rien à modifier : exécutez simplement la cellule.

In [ ]:
def verifier(nom, tests):
    """tests : liste de (fonction qui renvoie True/False, message si c'est faux)."""
    erreurs = []
    for test, message in tests:
        try:
            ok = test()
        except Exception:          # variable encore à None, mauvais type...
            ok = False
        if not ok:
            erreurs.append(message)
    if erreurs:
        print(f"❌ {nom} : à corriger")
        for m in erreurs:
            print("   -", m)
    else:
        print(f"✅ {nom} : tout est correct !")

### 0.3 Le corpus fil rouge
70 avis de restaurants, en français et en darija (écrite en lettres latines), avec leur sentiment.
On y retrouve les avis du cours (les dix premiers), plus d'autres avis avec du « bruit » : majuscules, émojis, URL, mention `@…`, lettres répétées (« Trooop »).

In [ ]:
# Le corpus fil rouge : 70 avis de restaurants (français + quelques avis en darija)
# Chaque élément est un tuple (texte de l'avis, sentiment)
CORPUS = [
    ('Service rapide et personnel souriant, je recommande ce restaurant.', 'positif'),
    ('Le tajine au poulet était délicieux, service rapide.', 'positif'),
    ('Très bon rapport qualité prix, le couscous était parfait.', 'positif'),
    ('Service lent, le plat était froid. Je ne recommande pas.', 'négatif'),
    ('Le tajine au poulet est excellent et pas cher.', 'positif'),
    ('Très bon accueil, service rapide, je recommande.', 'positif'),
    ('Le couscous était froid et le service lent.', 'négatif'),
    ('Rapport qualité prix correct, mais le service est lent.', 'négatif'),
    ('Personnel souriant, tajine au poulet délicieux, très bon moment.', 'positif'),
    ('Pas cher, très bon, je recommande ce restaurant à Rabat.', 'positif'),
    ('Trooop bon !!! Le meilleur couscous de Casablanca 😍', 'positif'),
    ("Attente beaucoup trop longue, plus d'une heure pour un tajine froid.", 'négatif'),
    ('Cadre agréable, terrasse ensoleillée et service rapide.', 'positif'),
    ('Les prix sont trop élevés pour des portions aussi petites.', 'négatif'),
    ("Le thé à la menthe est excellent, l'accueil chaleureux.", 'positif'),
    ('Service lent et serveur désagréable, je ne reviendrai pas.', 'négatif'),
    ('Très bon tajine au poulet aux olives, je recommande vivement.', 'positif'),
    ('La pastilla au poulet était sèche et pas assez chaude.', 'négatif'),
    ('Rapport qualité prix imbattable, personnel souriant et attentionné.', 'positif'),
    ('Plat froid, service lent, prix élevés : une vraie déception.', 'négatif'),
    ('Le couscous royal est généreux et le service rapide.', 'positif'),
    ('Réservation perdue, attente interminable, accueil froid.', 'négatif'),
    ('Très bon restaurant, la terrasse est agréable le soir.', 'positif'),
    ('Le tajine de viande était trop salé et la viande dure.', 'négatif'),
    ('Accueil chaleureux, thé à la menthe offert, merci !', 'positif'),
    ('Les portions sont petites mais le goût est excellent.', 'positif'),
    ('Toilettes sales et salle bruyante, je ne recommande pas ce restaurant.', 'négatif'),
    ('Très bon rapport qualité prix, le personnel est souriant.', 'positif'),
    ('Service rapide le midi, parfait pour une pause déjeuner.', 'positif'),
    ('Le serveur a oublié notre commande, attente très longue.', 'négatif'),
    ('La harira est délicieuse et pas chère, je recommande.', 'positif'),
    ('Plats trop gras et service lent, dommage.', 'négatif'),
    ('Le meilleur tajine au poulet de Marrakech, rien à dire.', 'positif'),
    ('Prix élevés, musique trop forte, service lent.', 'négatif'),
    ('Très bon accueil et cadre magnifique, on reviendra.', 'positif'),
    ('Le couscous aux sept légumes était fade et froid.', 'négatif'),
    ('Pas cher et copieux, très bon rapport qualité prix.', 'positif'),
    ('Commande en ligne sur https://resto-exemple.ma : livraison rapide et plat chaud.', 'positif'),
    ('Livraison en retard et plat froid, je ne commanderai plus.', 'négatif'),
    ('@chef_rabat bravo pour le tajine au poulet, un régal !', 'positif'),
    ('Le personnel est souriant mais le service est lent.', 'négatif'),
    ('Très bon thé à la menthe et pâtisseries délicieuses.', 'positif'),
    ('Salle sale, nappes tachées, une mauvaise expérience.', 'négatif'),
    ('Service rapide, plats chauds et personnel souriant.', 'positif'),
    ("Trop cher pour ce que c'est, je ne recommande pas.", 'négatif'),
    ('La terrasse est agréable et le service rapide, très bon moment.', 'positif'),
    ("Le poisson n'était pas frais, très déçu.", 'négatif'),
    ('Tajine au poulet et citron confit : un délice, je recommande.', 'positif'),
    ('Accueil froid et attente longue avant de commander.', 'négatif'),
    ('Très bon couscous le vendredi, ambiance familiale.', 'positif'),
    ('Les frites étaient froides et le burger sec.', 'négatif'),
    ('Rapport qualité prix excellent, je recommande ce restaurant à tous.', 'positif'),
    ('Service lent, serveurs débordés, prix élevés.', 'négatif'),
    ('Cuisine marocaine authentique, tajine au poulet parfait.', 'positif'),
    ('Pas de climatisation en plein été, salle trop chaude.', 'négatif'),
    ('Très bon accueil, plats copieux et pas chers.', 'positif'),
    ("Le plat est arrivé froid après quarante minutes d'attente.", 'négatif'),
    ('Personnel souriant et service rapide, rien à redire.', 'positif'),
    ('Viande trop cuite et sauce fade, je ne reviendrai pas.', 'négatif'),
    ('Le couscous était délicieux et le thé à la menthe parfait.', 'positif'),
    ('lmakla bnina bzaf w service rapide, nsah bih', 'positif'),
    ('tajine dial djaj bnin bzaf, prix mzyan', 'positif'),
    ('service bti2 bzaf w lmakla barda, ma nsahch', 'négatif'),
    ('lmakla bnina walakin ghalya chwiya', 'négatif'),
    ('couscous bnin bzaf, nsah bih, service mzyan', 'positif'),
    ('tsennina sa3a kamla, service bti2 w lmakla barda', 'négatif'),
    ('staff drif bzaf w lmakla bnina, nsah bih', 'positif'),
    ('ghali bzaf w lmakla ma bninach', 'négatif'),
    ('Très bon restaurant, service rapide et plats délicieux.', 'positif'),
    ('Le tajine était froid et le service lent, très déçu.', 'négatif'),
]

avis = [texte for texte, sentiment in CORPUS]            # les textes seuls
sentiments = [sentiment for texte, sentiment in CORPUS]  # les étiquettes seules

print(len(avis), "avis :", sentiments.count("positif"), "positifs et",
      sentiments.count("négatif"), "négatifs")
print("Exemple :", avis[10])

### 0.4 Prétraiter et tokeniser
`preprocess()` reprend en version courte le nettoyage du TP2 ; `tokeniser()` découpe en mots avec la regex `\w+`
(lettres, accents compris, et chiffres). Si vous avez votre propre `preprocess()` du TP2, vous pouvez la coller ici.

In [ ]:
def preprocess(texte):
    """Nettoyage simple (version courte du TP2)."""
    texte = texte.lower()                          # 1. minuscules
    texte = re.sub(r"https?://\S+", " ", texte)    # 2. retirer les URL
    texte = re.sub(r"@\w+", " ", texte)            # 3. retirer les mentions @...
    texte = re.sub(r"(\w)\1{2,}", r"\1", texte)    # 4. « trooop » -> « trop »
    return texte

def tokeniser(texte):
    """Liste des mots en minuscules, sans ponctuation ni émojis."""
    return re.findall(r"\w+", preprocess(texte))

print(tokeniser("Trooop bon !!! Le meilleur couscous 😍 https://x.ma @chef"))
print(tokeniser(avis[3]))

> 💡 Remarque : « n'était » donne deux tokens, `n` et `était`. La négation `n` est donc conservée comme un mot.

## Partie 1 · Compter : tokens, types, hapax, TTR

**Rappel du cours.**

| Mesure | Définition |
|---|---|
| Tokens (N) | toutes les occurrences de mots |
| Types (V) | les mots différents |
| Hapax | les types vus une seule fois |
| TTR | V / N : la richesse du vocabulaire (entre 0 et 1) |

**Exemple** sur la phrase du cours : exécutez la cellule et vérifiez les chiffres à la main.

In [ ]:
phrase = "le chat voit le chien . le chien dort ."
mots = [t for t in phrase.split() if t != "."]
frequences = Counter(mots)                  # compte chaque mot

print("tokens N :", len(mots))
print("types  V :", len(frequences))
print("hapax    :", [m for m, f in frequences.items() if f == 1])
print("TTR      :", round(len(frequences) / len(mots), 3))

### Exercice 1 · Les chiffres du corpus
1. `docs` : une liste qui contient, pour chaque avis, **la liste de ses tokens** (utilisez `tokeniser`).
2. `tokens` : **tous** les tokens du corpus dans une seule liste (on « aplatit » `docs`).
3. `freq` : le `Counter` des tokens.
4. `N`, `V`, `hapax` (liste des mots de fréquence 1) et `ttr`.

> 💡 Aplatir une liste de listes : `[x for sous_liste in liste for x in sous_liste]`

In [ ]:
docs = [tokeniser(texte) for texte in avis]
tokens = [mot for doc in docs for mot in doc]
freq = Counter(tokens)

N = len(tokens)
V = len(freq)
hapax = [mot for mot, f in freq.items() if f == 1]
ttr = V / N

print("N =", N, "| V =", V, "| hapax =", len(hapax), "| TTR =", round(ttr, 4))

In [ ]:
verifier("Exercice 1", [
    (lambda: len(docs) == 70, "docs doit contenir 70 listes (une par avis)"),
    (lambda: docs[3] == tokeniser(avis[3]), "docs[3] doit être tokeniser(avis[3])"),
    (lambda: N == 605, "N doit valoir 605 (nombre total de tokens)"),
    (lambda: V == 197, "V doit valoir 197 (nombre de types)"),
    (lambda: len(hapax) == 116, "il doit y avoir 116 hapax"),
    (lambda: abs(ttr - V / N) < 1e-9, "ttr doit valoir V / N"),
])

**Question 1.** Que vaut le TTR ? Plus de la moitié des types sont des hapax : est-ce surprenant ?

✍️ **Réponse 1.** Le TTR vaut 197 / 605 ≈ **0,326**. Les 116 hapax représentent 58,9 % des 197 types : c'est plausible dans un petit corpus mêlant avis français, darija en caractères latins, noms de plats et variantes orthographiques. Le TTR mesure ici la diversité des formes observées, pas directement la qualité du vocabulaire.

### Exercice 2 · Le TTR dépend de la longueur du texte
Calculez le TTR des **100, 200, 400 premiers tokens**, puis de tout le corpus, et rangez les valeurs dans la liste `ttrs`.

> 💡 `tokens[:100]` donne les 100 premiers tokens ; `set(liste)` donne les éléments différents.

In [ ]:
ttrs = []
for taille in [100, 200, 400, N]:
    extrait = tokens[:taille]
    ttr_extrait = len(set(extrait)) / len(extrait)
    ttrs.append(ttr_extrait)
    print(f"{taille:4} tokens -> TTR = {ttr_extrait:.3f}")

In [ ]:
verifier("Exercice 2", [
    (lambda: len(ttrs) == 4, "ttrs doit contenir 4 valeurs"),
    (lambda: abs(ttrs[0] - len(set(tokens[:100])) / 100) < 1e-9, "TTR des 100 premiers tokens incorrect"),
    (lambda: ttrs[0] > ttrs[1] > ttrs[2] > ttrs[3], "le TTR doit diminuer quand le texte s'allonge"),
])

**Question 2.** Pourquoi le TTR baisse-t-il quand le texte s'allonge ? Quelle précaution faut-il prendre pour comparer deux corpus ?

✍️ **Réponse 2.** Le TTR baisse de 0,440 (100 tokens) à 0,326 (605 tokens), car les mots déjà vus reviennent au fur et à mesure. Pour comparer deux corpus, il faut employer des échantillons de même longueur et la même tokenisation ; sinon le corpus plus long est désavantagé.

## Partie 2 · Fréquences et mots vides

**Rappel.** `freq.most_common(k)` renvoie les k mots les plus fréquents sous forme de couples `(mot, fréquence)`.
La **fréquence relative** f / N permet de comparer des corpus de tailles différentes.

**Exemple :** affichage aligné avec les f-strings (`{mot:10}` réserve 10 caractères, `{x:.1%}` affiche un pourcentage).

In [ ]:
for mot, f in Counter(["le", "chat", "le", "chien", "le"]).most_common():
    print(f"{mot:10} {f:2}  {f / 5:.1%}")

### Exercice 3 · Les 15 mots les plus fréquents
Affichez les 15 mots les plus fréquents du corpus avec leur fréquence absolue et leur fréquence relative.

In [ ]:
top15 = freq.most_common(15)
for mot, f in top15:
    print(f"{mot:12} {f:3}  {f / N:.1%}")

In [ ]:
verifier("Exercice 3", [
    (lambda: len(top15) == 15, "top15 doit contenir 15 couples"),
    (lambda: top15[0] == ("le", 31), "le mot le plus fréquent est « le » (31 fois)"),
    (lambda: top15[2][0] == "service", "le 3e mot doit être « service »"),
])

**Question 3.** Quels types de mots sont en tête ? Quel est le premier mot « plein » et que dit-il du corpus ?

✍️ **Réponse 3.** Les premiers mots sont surtout grammaticaux : `le` (31) et `et` (29). Le premier mot plein est `service` (24) : il montre que les avis évaluent beaucoup l'accueil et la rapidité du restaurant. La fréquence seule ne dit pas si cette évaluation est positive ou négative.

### Exercice 4 · Retirer les mots vides généraux
La liste française de NLTK contient 157 mots vides… dont **la négation** (`ne`, `pas`, `n`). Pour des avis, il faut la **garder** : elle porte le sentiment (« je ne recommande pas »).

1. `mots_vides` : l'ensemble des mots vides de NLTK **moins** `{"ne", "pas", "n"}`.
2. `tokens_pleins` : les tokens qui ne sont pas des mots vides.
3. `taux` : la part des tokens retirés (entre 0 et 1).
4. `freq_pleins` : le `Counter` de `tokens_pleins`.

> 💡 Différence de deux ensembles : `A - B`. Test d'appartenance : `mot not in A` (rapide sur un `set`).

In [ ]:
liste_nltk = stopwords.words("french")
print(len(liste_nltk), "mots vides, par exemple :", liste_nltk[:12])
print("« pas » est dans la liste ?", "pas" in liste_nltk)

In [ ]:
negation = {"ne", "pas", "n"}
mots_vides = set(stopwords.words("french")) - negation
tokens_pleins = [mot for mot in tokens if mot not in mots_vides]
taux = 1 - len(tokens_pleins) / N
freq_pleins = Counter(tokens_pleins)

print(f"Tokens retirés : {taux:.1%}")
print(freq_pleins.most_common(10))

In [ ]:
verifier("Exercice 4", [
    (lambda: isinstance(mots_vides, set), "mots_vides doit être un set"),
    (lambda: "pas" not in mots_vides and "ne" not in mots_vides, "la négation (ne, pas, n) doit être gardée"),
    (lambda: "le" in mots_vides, "« le » doit rester un mot vide"),
    (lambda: len(tokens_pleins) == 435, "tokens_pleins doit contenir 435 tokens"),
    (lambda: abs(taux - (1 - 435 / 605)) < 1e-9, "taux = 1 - len(tokens_pleins) / N"),
    (lambda: freq_pleins.most_common(1)[0][0] == "service", "après filtrage, le 1er mot est « service »"),
])

**Question 4.** Quelle part du texte les mots vides représentent-ils ? Qu'est-ce que cela confirme ?

✍️ **Réponse 4.** Les mots vides généraux représentent 170 / 605 ≈ **28,1 %** des tokens. Ils occupent une grande place dans le texte ; leur retrait facilite l'exploration lexicale. La négation (`ne`, `n`, `pas`) est conservée, car elle change le sens d'un avis.

### Exercice 5 · Les mots vides de domaine
Un **mot vide de domaine** apparaît dans une grande partie des documents **de ce corpus** : il ne permet plus de les distinguer.
On le repère par sa **fréquence documentaire** df = nombre d'avis qui contiennent le mot (au moins une fois).

**Exemple** sur trois mini-documents : `set(doc)` compte chaque mot une seule fois par document.

In [ ]:
mini = [["le", "service", "rapide"], ["le", "service", "service", "lent"], ["le", "tajine"]]
df_mini = Counter(mot for doc in mini for mot in set(doc))
print(df_mini)      # « service » : présent dans 2 documents (même s'il apparaît 3 fois)

1. `df` : la fréquence documentaire de chaque mot du corpus.
2. `candidats` : les mots présents dans **au moins 15 %** des avis **et** qui ne sont pas dans `mots_vides`, du plus au moins fréquent.
3. Choisissez ensuite `mots_vides_domaine` : les candidats qui ne portent **ni opinion ni négation**.

In [ ]:
df = Counter(mot for doc in docs for mot in set(doc))
seuil = 0.15
candidats = sorted(
    (mot for mot, nombre in df.items()
     if nombre / len(docs) >= seuil and mot not in mots_vides),
    key=lambda mot: (-df[mot], mot),
)

for mot in candidats:
    print(f"{mot:10} présent dans {df[mot]:2} avis sur {len(docs)} ({df[mot] / len(docs):.0%})")

mots_vides_domaine = {"service"}

In [ ]:
verifier("Exercice 5", [
    (lambda: df["service"] == 24, "df['service'] doit valoir 24"),
    (lambda: df["le"] == 26, "df['le'] doit valoir 26 (et non 31 : on compte une fois par avis)"),
    (lambda: candidats[0] == "service", "le premier candidat doit être « service »"),
    (lambda: "le" not in candidats, "les mots vides généraux ne sont pas des candidats"),
    (lambda: "service" in mots_vides_domaine, "« service » est un mot vide de domaine"),
    (lambda: not ({"bon", "rapide", "pas", "très"} & mots_vides_domaine), "ne retirez pas les mots d'opinion ni la négation"),
])

**Question 5.** Pourquoi ne faut-il pas retirer « bon », « rapide » ou « pas », alors qu'ils sont aussi très fréquents ?

✍️ **Réponse 5.** `bon` et `rapide` indiquent directement une appréciation positive, même s'ils sont fréquents. `pas` peut inverser une opinion (`pas bon`) ou participer à une expression positive (`pas cher`). Les supprimer ferait perdre des indices essentiels pour l'analyse de sentiment.

## Partie 3 · La loi de Zipf

**Rappel.** Si l'on classe les mots du plus fréquent au moins fréquent, la fréquence f d'un mot est à peu près inversement proportionnelle à son rang r :
**f × r ≈ C**. En échelle log-log, log f = log C − log r : une **droite de pente −1**.

**Exemple :** `enumerate(liste, start=1)` numérote les éléments à partir de 1, ce qui donne directement le rang.

In [ ]:
for r, (mot, f) in enumerate([("le", 8), ("chat", 4), ("chien", 3)], start=1):
    print(r, mot, f, "-> r × f =", r * f)

### Exercice 6 · Le tableau rang × fréquence
Pour les rangs 1, 2, 3, 5, 10, 20, 50 et 100, affichez le mot, sa fréquence et le produit r × f.
Rangez les produits dans le dictionnaire `produits` (clé : rang, valeur : r × f).

In [ ]:
classement = freq.most_common()
produits = {}
for r, (mot, f) in enumerate(classement, start=1):
    if r in (1, 2, 3, 5, 10, 20, 50, 100):
        produits[r] = r * f
        print(f"{r:4}  {mot:12} {f:4}  {produits[r]}")

In [ ]:
verifier("Exercice 6", [
    (lambda: sorted(produits) == [1, 2, 3, 5, 10, 20, 50, 100], "il faut les 8 rangs demandés"),
    (lambda: produits[1] == 31 and produits[2] == 58, "r × f : 31 au rang 1, 58 au rang 2"),
])

### Exercice 7 · La courbe de Zipf en log-log
Tracez les fréquences en fonction du rang avec `plt.loglog`, puis la droite théorique **f = C / r** avec C = f₁ (la fréquence du mot n° 1).
Enregistrez l'image sous `zipf_avis.png`.

In [ ]:
f = [nombre for _, nombre in classement]
rangs = list(range(1, len(f) + 1))
zipf = [f[0] / r for r in rangs]

plt.figure(figsize=(6, 4))
plt.loglog(rangs, f, ".", label="Corpus", color="#1768ac")
plt.loglog(rangs, zipf, "--", label="f₁ / r", color="#d05f26")
plt.xlabel("rang r (échelle log)")
plt.ylabel("fréquence f (échelle log)")
plt.title("Loi de Zipf sur les 70 avis")
plt.legend()
plt.grid(True, which="both", alpha=0.2)
plt.savefig("zipf_avis.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
verifier("Exercice 7", [
    (lambda: list(f[:3]) == [31, 29, 24], "f doit commencer par 31, 29, 24"),
    (lambda: len(f) == V and list(rangs)[-1] == V, "il faut un rang par type (1 à V)"),
    (lambda: abs(zipf[9] - 3.1) < 1e-9, "zipf[9] doit valoir f[0] / 10 = 3,1"),
])

**Question 6.** Les points suivent-ils la droite ? Que représentent les « marches » horizontales en bas à droite ?

✍️ **Réponse 6.** Les fréquences décroissent avec le rang, mais les points restent nettement au-dessus de la droite théorique `f₁/r` dès les premiers rangs : l'ajustement est seulement qualitatif sur ce petit corpus. Les marches horizontales à droite regroupent des mots de même fréquence, surtout les 116 hapax de fréquence 1 : la fréquence est un entier, et de nombreux rangs ont donc la même valeur.

### Exercice 8 · La tête et la traîne
Calculez :
- `part_top10` : la part du texte (des tokens) couverte par les **10 mots les plus fréquents** ;
- `part_hapax` : la part des **types** qui sont des hapax.

In [ ]:
part_top10 = sum(nombre for _, nombre in freq.most_common(10)) / N
part_hapax = len(hapax) / V
print(f"Les 10 premiers mots couvrent {part_top10:.1%} du texte")
print(f"Les hapax représentent {part_hapax:.1%} des types")

In [ ]:
verifier("Exercice 8", [
    (lambda: abs(part_top10 - 179 / 605) < 1e-9, "part_top10 = somme des 10 plus grandes fréquences / N"),
    (lambda: abs(part_hapax - 116 / 197) < 1e-9, "part_hapax = nombre d'hapax / V"),
])

## Partie 4 · Les n-grammes

**Rappel.** Un n-gramme est une suite de n mots consécutifs. Avec une liste de N mots, il y a **N − n + 1** n-grammes.
On les calcule **avis par avis**, pour ne pas créer de bigramme « à cheval » entre deux avis.

**Exemple :** le découpage `liste[i:i+n]` prend n éléments à partir de la position i.

In [ ]:
mots = ["le", "service", "est", "très", "rapide"]
print(mots[1:3])                                   # ['service', 'est']
print([tuple(mots[i:i+2]) for i in range(4)])      # les 4 bigrammes

### Exercice 9 · La fonction `ngrammes`
Écrivez `ngrammes(tokens, n)` qui renvoie la **liste des n-grammes** (des tuples) de la liste `tokens`.

In [ ]:
def ngrammes(tokens, n):
    if n < 1:
        raise ValueError("n doit être positif")
    return [tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]

print(ngrammes(["le", "service", "est", "très", "rapide"], 2))

In [ ]:
verifier("Exercice 9", [
    (lambda: ngrammes(["a", "b", "c"], 2) == [("a", "b"), ("b", "c")], "ngrammes(['a','b','c'], 2) doit donner [('a','b'), ('b','c')]"),
    (lambda: len(ngrammes(list("abcdefg"), 3)) == 5, "7 mots -> 7 - 3 + 1 = 5 trigrammes"),
    (lambda: ngrammes(["a"], 2) == [], "une liste trop courte donne une liste vide"),
])

### Exercice 10 · Les bigrammes et trigrammes fréquents
Comptez les bigrammes et les trigrammes **avis par avis** dans deux `Counter`, puis affichez les 10 bigrammes et les 5 trigrammes les plus fréquents.

In [ ]:
bigrammes = Counter(b for doc in docs for b in ngrammes(doc, 2))
trigrammes = Counter(t for doc in docs for t in ngrammes(doc, 3))

print("10 bigrammes les plus fréquents :")
for b, nombre in bigrammes.most_common(10):
    print(f"{' '.join(b):24} {nombre}")
print("5 trigrammes les plus fréquents :")
for t, nombre in trigrammes.most_common(5):
    print(f"{' '.join(t):24} {nombre}")

In [ ]:
verifier("Exercice 10", [
    (lambda: sum(bigrammes.values()) == N - len(docs), "il doit y avoir N - 70 = 535 bigrammes (calcul avis par avis)"),
    (lambda: bigrammes.most_common(1)[0] == (("très", "bon"), 14), "le bigramme le plus fréquent est « très bon » (14)"),
    (lambda: trigrammes[("tajine", "au", "poulet")] == 8, "« tajine au poulet » apparaît 8 fois"),
])

**Question 7.** Parmi les bigrammes fréquents, lesquels sont de vraies expressions ? Lesquels sont des suites banales ? Pourquoi la fréquence seule ne suffit-elle pas ?

✍️ **Réponse 7.** `très bon`, `service rapide`, `service lent` et `rapport qualité` sont des associations utiles ; `tajine au poulet` décrit un plat. `et le` est une suite grammaticale banale, pourtant fréquente. Le comptage seul privilégie les mots courants : une mesure d'association et l'examen du contexte sont nécessaires.

## Partie 5 · Les collocations : PMI et LLR

**Rappel.**
- **PMI** (force de l'association) : PMI(x, y) = log₂ ( P(x, y) / (P(x) × P(y)) ), avec P(x) = c(x) / N et P(x, y) = c(x, y) / N.
- **LLR** ou G² (fiabilité de l'association) : G² = 2 × Σ O × ln(O / E) sur les 4 cases du tableau de contingence.

**Exemple :** `math.log2` (log en base 2, pour la PMI) et `math.log` (log népérien ln, pour le LLR).

In [ ]:
print(math.log2(8))             # 3.0 car 2³ = 8
print(round(math.log(2.718281828), 3))   # ≈ 1 car ln(e) = 1

### Exercice 11 · La PMI à la main, puis en Python
Avec N = 1 000, c(pas) = 20, c(cher) = 10 et c(pas cher) = 8 :
P(pas) = 0,020 ; P(cher) = 0,010 ; P(pas, cher) = 0,008.
Le rapport observé / attendu vaut 0,008 / (0,020 × 0,010) = **40** ; la PMI vaut log₂(40) ≈ **5,322 bits**.

La fonction ci-dessous vérifie ce calcul et l'exemple « service rapide » du cours.

In [ ]:
def pmi(c_xy, c_x, c_y, N):
    if min(c_xy, c_x, c_y, N) <= 0:
        raise ValueError("les quatre effectifs doivent être positifs")
    return math.log2(c_xy * N / (c_x * c_y))

print("pas cher       :", pmi(8, 20, 10, 1000))
print("service rapide :", pmi(20, 100, 50, 10_000))

In [ ]:
verifier("Exercice 11", [
    (lambda: abs(pmi(8, 20, 10, 1000) - math.log2(40)) < 1e-9, "pmi(8, 20, 10, 1000) doit valoir log2(40) ≈ 5,32"),
    (lambda: abs(pmi(2, 2, 2, 15) - math.log2(7.5)) < 1e-9, "pmi(2, 2, 2, 15) doit valoir log2(7,5) ≈ 2,91 (exemple « thé fort »)"),
    (lambda: abs(pmi(1, 10, 10, 100)) < 1e-9, "deux mots indépendants donnent une PMI nulle"),
])

### Exercice 12 · La PMI sur le corpus
Calculez la PMI de tous les bigrammes vus **au moins 3 fois** (dictionnaire `scores_pmi`), puis affichez les 10 meilleurs.
Utilisez `freq` pour c(x) et c(y), `bigrammes` pour c(x, y) et `N` pour le nombre de tokens.

> 💡 Trier un dictionnaire par valeur décroissante : `sorted(d.items(), key=lambda kv: kv[1], reverse=True)`

In [ ]:
seuil_freq = 3
scores_pmi = {
    bigramme: pmi(nombre, freq[bigramme[0]], freq[bigramme[1]], N)
    for bigramme, nombre in bigrammes.items() if nombre >= seuil_freq
}
top_pmi = sorted(scores_pmi.items(), key=lambda element: (-element[1], element[0]))[:10]

for (x, y), score in top_pmi:
    print(f"{x + ' ' + y:22} c={bigrammes[(x, y)]:2}  PMI={score:.2f}")

In [ ]:
verifier("Exercice 12", [
    (lambda: all(bigrammes[b] >= 3 for b in scores_pmi), "seuls les bigrammes vus au moins 3 fois sont gardés"),
    (lambda: len(scores_pmi) == 37, "il doit y avoir 37 bigrammes avec c >= 3"),
    (lambda: len(top_pmi) == 10 and top_pmi[0][1] >= top_pmi[-1][1], "top_pmi : 10 couples triés du meilleur au moins bon"),
    (lambda: scores_pmi[("service", "rapide")] > scores_pmi[("le", "service")], "« service rapide » doit battre « le service »"),
])

**Question 8.** Pourquoi faut-il un seuil de fréquence ? Que se passerait-il sans lui ?

✍️ **Réponse 8.** Sans seuil, deux mots rares qui se rencontrent une seule fois peuvent obtenir une PMI très élevée par hasard. Le seuil de trois occurrences limite cet effet, mais écarte aussi des expressions intéressantes peu fréquentes ; il doit être adapté à la taille du corpus.

### Exercice 13 · Le LLR (G²) pas à pas
Pour une paire (x, y), on remplit le tableau 2 × 2, on calcule l'attendu E de chaque case (total ligne × total colonne / N), puis
**G² = 2 × Σ O × ln(O / E)** (avec la convention 0 × ln 0 = 0).

| Case | O (observé) | E (attendu) |
|---|---|---|
| 11 | c(x, y) | c(x) × c(y) / N |
| 12 | c(x) − c(x, y) | c(x) × (N − c(y)) / N |
| 21 | c(y) − c(x, y) | (N − c(x)) × c(y) / N |
| 22 | N − c(x) − c(y) + c(x, y) | (N − c(x)) × (N − c(y)) / N |

**✍️ Vérifiez d'abord le cours :** pour « le chat » (c(le) = 3, c(chat) = 2, c(le, chat) = 2, N = 10), G² ≈ 6,19.

In [ ]:
def llr(c_xy, c_x, c_y, N):
    if N <= 0 or min(c_xy, c_x - c_xy, c_y - c_xy, N - c_x - c_y + c_xy) < 0:
        raise ValueError("effectifs incohérents")
    O = [c_xy, c_x - c_xy, c_y - c_xy, N - c_x - c_y + c_xy]
    E = [c_x * c_y / N, c_x * (N - c_y) / N,
         (N - c_x) * c_y / N, (N - c_x) * (N - c_y) / N]
    return 2 * sum(o * math.log(o / e) for o, e in zip(O, E) if o > 0)

print("le chat   :", llr(2, 3, 2, 10))
print("chat noir :", llr(1, 2, 1, 10))

In [ ]:
verifier("Exercice 13", [
    (lambda: abs(llr(2, 3, 2, 10) - 6.19) < 0.01, "llr(2, 3, 2, 10) doit valoir 6,19 (exemple « le chat »)"),
    (lambda: abs(llr(1, 2, 1, 10) - 3.73) < 0.01, "llr(1, 2, 1, 10) doit valoir 3,73 (exemple « chat noir »)"),
    (lambda: abs(llr(1, 10, 10, 100)) < 1e-9, "deux mots indépendants donnent un LLR nul"),
])

Classez maintenant les bigrammes vus **au moins 2 fois** par LLR décroissant, et comparez avec la PMI.

In [ ]:
scores_llr = {
    bigramme: llr(nombre, freq[bigramme[0]], freq[bigramme[1]], N)
    for bigramme, nombre in bigrammes.items() if nombre >= 2
}
top_llr = sorted(scores_llr.items(), key=lambda element: (-element[1], element[0]))[:10]

for (x, y), score in top_llr:
    print(f"{x + ' ' + y:22} c={bigrammes[(x, y)]:2}  LLR={score:6.2f}")

In [ ]:
verifier("Exercice 13 (suite)", [
    (lambda: all(bigrammes[b] >= 2 for b in scores_llr), "seuls les bigrammes vus au moins 2 fois sont gardés"),
    (lambda: top_llr[0][0] == ("très", "bon"), "le meilleur LLR est « très bon »"),
    (lambda: len(top_llr) == 10, "top_llr doit contenir 10 couples"),
])

**Question 9.** Comparez les deux classements. Quelle mesure favorise les paires fréquentes ? Laquelle les paires exclusives ?

✍️ **Réponse 9.** La PMI met en tête des paires presque exclusives, parfois peu fréquentes (`nsah bih`, `lmakla bnina`). Le LLR favorise les associations à la fois nettes et répétées : `très bon` (14 occurrences) arrive premier. Les deux mesures sont complémentaires et leur classement dépend aussi du seuil de fréquence choisi.

### Exercice 14 · Les collocations avec NLTK
`BigramCollocationFinder` compte les unigrammes et les bigrammes à notre place.
On ajoute un jeton `<FIN>` après chaque avis pour ne pas créer de bigramme entre deux avis, puis on l'ignore avec `apply_word_filter`.

| Méthode | Rôle |
|---|---|
| `BigramCollocationFinder.from_words(liste)` | compte unigrammes et bigrammes |
| `finder.apply_word_filter(f)` | retire les bigrammes qui contiennent un mot tel que f(mot) est vrai |
| `finder.apply_freq_filter(k)` | garde les bigrammes vus au moins k fois |
| `finder.nbest(mesure, k)` | les k meilleurs bigrammes |
| `finder.score_ngrams(mesure)` | tous les bigrammes avec leur score |

In [ ]:
tokens_fin = [mot for doc in docs for mot in [*doc, "<FIN>"]]
mesures = BigramAssocMeasures()
finder = BigramCollocationFinder.from_words(tokens_fin)
finder.apply_word_filter(lambda mot: mot == "<FIN>")
finder.apply_freq_filter(3)

nltk_pmi = finder.nbest(mesures.pmi, 10)
nltk_llr = finder.nbest(mesures.likelihood_ratio, 10)
print("PMI :", nltk_pmi)
print("LLR :", nltk_llr)

In [ ]:
verifier("Exercice 14", [
    (lambda: tokens_fin.count("<FIN>") == 70, "tokens_fin doit contenir un <FIN> par avis"),
    (lambda: len(nltk_pmi) == 10 and len(nltk_llr) == 10, "il faut 10 bigrammes par mesure"),
    (lambda: all("<FIN>" not in b for b in nltk_pmi + nltk_llr), "aucun bigramme ne doit contenir <FIN>"),
    (lambda: nltk_llr[0] == ("très", "bon"), "le meilleur LLR selon NLTK est « très bon »"),
])

**Question 10.** Les listes de NLTK ressemblent-elles aux vôtres ? D'où peuvent venir les petites différences de scores ?

✍️ **Réponse 10.** Avec le seuil de trois occurrences, les dix paires et leur ordre coïncident ici entre NLTK et notre calcul, pour la PMI comme pour le LLR. De petites différences numériques peuvent venir du dénominateur : le `BigramCollocationFinder` voit les 70 marqueurs `<FIN>` ajoutés au flux, même si les paires qui les contiennent sont filtrées. Les conventions de comptage et les arrondis jouent aussi.

## Partie 6 · Les concordances (KWIC)

**Rappel.** Une concordance affiche toutes les occurrences d'un mot, **alignées au centre**, avec leur contexte gauche et droit.

**Exemple :** `re.finditer` donne chaque occurrence avec sa position (`m.start()`, `m.end()`) ; `\b` impose un mot entier ; `{texte:>20}` aligne à droite sur 20 caractères.

In [ ]:
texte = "Service rapide, mais le service du soir est lent."
for m in re.finditer(r"\bservice\b", texte, flags=re.IGNORECASE):
    print(m.start(), m.end(), repr(m.group()), "| à gauche :", repr(texte[:m.start()][-12:]))
print(f"[{'droite':>10}]  [{'gauche':<10}]")

### Exercice 15 · Votre propre concordance
Écrivez `concordance(textes, mot, largeur=30)` qui, pour chaque avis, trouve toutes les occurrences du **mot entier** (sans tenir compte des majuscules)
et **renvoie la liste des lignes** au format : `contexte gauche aligné à droite` + ` [mot] ` + `contexte droit`.

In [ ]:
def concordance(textes, mot, largeur=30):
    lignes = []
    for texte in textes:
        for m in re.finditer(None, texte, flags=re.IGNORECASE):   # À COMPLÉTER : le motif
            gauche = None    # À COMPLÉTER : au plus `largeur` caractères avant le mot
            droite = None    # À COMPLÉTER : au plus `largeur` caractères après le mot
            lignes.append(f"{gauche:>{largeur}} [{m.group()}] {droite}")
    return lignes

for ligne in concordance(avis, "service")[:8]:
    print(ligne)

In [ ]:
verifier("Exercice 15", [
    (lambda: len(concordance(avis, "service")) == 24, "« service » apparaît 24 fois dans les avis"),
    (lambda: len(concordance(["Le Service et les services"], "service")) == 1, "seul le mot entier compte (pas « services »)"),
    (lambda: concordance(["le service rapide"], "service", 5)[0] == "  le  [service]  rapi", "format : gauche aligné à droite, [mot], puis la droite"),
])

### Exercice 16 · Les concordances avec NLTK
`nltk.Text(liste_de_tokens)` crée un texte NLTK ; ses méthodes :
- `t.concordance(mot, width=79, lines=25)` **affiche** les lignes ;
- `t.concordance_list(mot, width=79, lines=25)` **renvoie** les lignes : chaque ligne a `.left` (mots à gauche), `.query` (le mot) et `.right` (mots à droite).

1. Affichez la concordance de « service » (largeur 60, 6 lignes).
2. Avec `concordance_list`, comptez le **premier mot à droite** de « service » (`droite_service`) puis de « pas » (`droite_pas`).

In [ ]:
t = None    # À COMPLÉTER : un nltk.Text de tous les tokens
# À COMPLÉTER : afficher la concordance de « service »

res = None  # À COMPLÉTER : concordance_list de « service » (lines=100 pour tout récupérer)
droite_service = None   # À COMPLÉTER : Counter du premier mot à droite
droite_pas = None       # À COMPLÉTER : même chose pour « pas »
print(droite_service)
print(droite_pas)

In [ ]:
verifier("Exercice 16", [
    (lambda: isinstance(t, Text), "t doit être un nltk.Text"),
    (lambda: len(res) == 24, "res doit contenir les 24 occurrences de « service »"),
    (lambda: droite_service["rapide"] == 11 and droite_service["lent"] == 8, "à droite de « service » : rapide 11 fois, lent 8 fois"),
    (lambda: droite_pas["cher"] == 3, "« pas cher » apparaît 3 fois"),
])

**Question 11.** Que révèle la concordance de « service » ? Et celle de « pas » : ce mot est-il toujours négatif ?

✍️ **Votre réponse :** *…*

## Partie 8 · Synthèse : l'étude descriptive du corpus

### Exercice 18 · Les chiffres clés en une fonction
Écrivez `chiffres_cles(docs)` qui renvoie un **dictionnaire** avec : `nb_avis`, `N`, `V`, `TTR`, `hapax` (nombre) et `longueur_moyenne` (nombre moyen de tokens par avis).
La fonction doit marcher pour **n'importe quelle** liste de documents tokenisés (elle servira dans la partie ★).

In [ ]:
def chiffres_cles(docs):
    # À COMPLÉTER
    return {}

for cle, valeur in chiffres_cles(docs).items():
    print(f"{cle:17} {valeur}")

In [ ]:
verifier("Exercice 18", [
    (lambda: chiffres_cles(docs)["N"] == 605 and chiffres_cles(docs)["V"] == 197, "N = 605 et V = 197"),
    (lambda: chiffres_cles(docs)["hapax"] == 116, "116 hapax"),
    (lambda: abs(chiffres_cles(docs)["longueur_moyenne"] - 8.6) < 0.06, "longueur moyenne ≈ 8,6 tokens par avis"),
    (lambda: chiffres_cles([["a", "b"], ["a"]])["V"] == 2, "la fonction doit marcher sur d'autres documents"),
])

### La synthèse (livrable)
Rédigez, dans la cellule ci-dessous, une courte étude descriptive du corpus (10 à 15 lignes) en vous appuyant sur vos résultats :
1. **Chiffres clés** : nombre d'avis, N, V, TTR, hapax ; ce que cela dit du vocabulaire.
2. **Fréquences et Zipf** : les mots en tête, les mots vides de domaine retenus, la forme de la courbe (insérez `zipf_avis.png`).
3. **10 collocations commentées** : expression, mesure (PMI ou LLR), ce qu'elle dit des avis.
4. **Concordances** : ce que révèlent « service » et « pas ».
5. **Limites** : taille du corpus, darija, prétraitement.

✍️ **Ma synthèse :**

*…*